# concateFile.ipynb

Notebook for advanced ECG concatenation with sequential HR ordering, pure/mixed level segments, and duration-aware truncation.

## Requirements implemented
- Sequential HR ordering (ascending HR number)
- Pure level segments (single label)
- Mixed level segments with majority-vote labels (ties -> lowest label)
- Duration-aware concatenation with smart truncation
- Time continuity preservation and column cleanup

In [45]:
import hashlib
import os
import random
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

import pandas as pd

In [46]:
class ECGAdvancedConcatenator:
    """
    Advanced ECG data concatenator with sequential HR ordering and duration-aware truncation.
    """

    def __init__(self, csv_label_file: Optional[str], data_dir: str, labels: Optional[List[int]] = None) -> None:
        if csv_label_file is not None and not os.path.isfile(csv_label_file):
            raise FileNotFoundError(f"CSV label file not found: {csv_label_file}")
        if not os.path.isdir(data_dir):
            raise FileNotFoundError(f"Data directory not found: {data_dir}")

        self.csv_label_file = csv_label_file
        self.data_dir = data_dir
        self.labels = labels or [0, 1, 2, 3]
        self.label_files: Dict[int, List[str]] = {}
        self.data_cache: Dict[str, pd.DataFrame] = {}

        if self.csv_label_file:
            self._load_label_mapping()
        else:
            self._scan_label_directories()

    def _scan_label_directories(self) -> None:
        for label in self.labels:
            label_dir = os.path.join(self.data_dir, str(label))
            if not os.path.isdir(label_dir):
                raise FileNotFoundError(f"Label directory not found: {label_dir}")
            files = [f for f in os.listdir(label_dir) if f.lower().endswith(".csv")]
            if not files:
                raise FileNotFoundError(f"No CSV files found in {label_dir}")
            self.label_files[label] = sorted(files, key=self._extract_hr_number)

    def _load_label_mapping(self) -> None:
        df = pd.read_csv(self.csv_label_file)
        df.columns = [c.strip() for c in df.columns]
        if "File" not in df.columns or "Label" not in df.columns:
            raise ValueError("CSV must include 'File' and 'Label' columns.")

        for _, row in df.iterrows():
            label = int(row["Label"])
            filename = str(row["File"])
            self.label_files.setdefault(label, []).append(filename)

        for label in self.label_files:
            self.label_files[label] = sorted(self.label_files[label], key=self._extract_hr_number)

    def _get_full_path(self, label: int, filename: str) -> str:
        return os.path.join(self.data_dir, str(label), filename)

    def _load_label_files(self, label: int) -> None:
        if label not in self.label_files:
            raise ValueError(f"Label {label} is not available.")

        for filename in self.label_files[label]:
            full_path = self._get_full_path(label, filename)
            if full_path in self.data_cache:
                continue
            if not os.path.isfile(full_path):
                raise FileNotFoundError(f"File missing for label {label}: {full_path}")
            df = pd.read_csv(full_path)
            df.columns = [c.strip() for c in df.columns]
            self.data_cache[full_path] = df

    @staticmethod
    def _get_duration_from_dataframe(df: pd.DataFrame) -> float:
        if "Time" in df.columns:
            time_series = df["Time"].to_numpy()
            if len(time_series) == 0:
                return 0.0
            return float(time_series[-1] - time_series[0])
        return float(len(df))

    @staticmethod
    def _extract_hr_number(filename: str) -> int:
        base = os.path.splitext(os.path.basename(filename))[0].lower()
        # Support names like hr80.csv and hr80_1.csv by reading the first hr<number> token.
        m = re.search(r"hr(\d+)", base)
        if m:
            return int(m.group(1))

        # Fallback: only parse the first numeric token before '_' to avoid hr80_1 -> 801.
        first_token = base.split("_", 1)[0]
        m2 = re.search(r"(\d+)", first_token)
        if m2:
            return int(m2.group(1))

        raise ValueError(f"Unable to extract HR number from filename: {filename}")

    @staticmethod
    def _offset_time(df: pd.DataFrame, offset: float) -> pd.DataFrame:
        if "Time" not in df.columns:
            return df
        df = df.copy()
        df["Time"] = df["Time"] + offset
        return df

    @staticmethod
    def _get_next_file_index(directory: str, prefix: str) -> int:
        max_index = 0
        if os.path.isdir(directory):
            for name in os.listdir(directory):
                if not name.lower().endswith(".csv"):
                    continue
                stem = os.path.splitext(name)[0]
                token = f"{prefix}_"
                if not stem.startswith(token):
                    continue
                suffix = stem[len(token):]
                if suffix.isdigit():
                    max_index = max(max_index, int(suffix))
        return max_index + 1

    @staticmethod
    def _hash_dataframe(df: pd.DataFrame) -> str:
        csv_bytes = df.to_csv(index=False).encode("utf-8")
        return hashlib.sha256(csv_bytes).hexdigest()

    @staticmethod
    def _collect_hashes_in_folder(folder: str) -> set:
        hashes = set()
        if not os.path.isdir(folder):
            return hashes
        for name in os.listdir(folder):
            if not name.lower().endswith(".csv"):
                continue
            fp = os.path.join(folder, name)
            with open(fp, "rb") as fh:
                hashes.add(hashlib.sha256(fh.read()).hexdigest())
        return hashes

    def concatenate_preserve_time(self, label: int, duration_minutes: float, random_order: bool = True) -> pd.DataFrame:
        self._load_label_files(label)
        label_files = list(self.label_files[label])
        if random_order:
            random.shuffle(label_files)
        else:
            label_files = sorted(label_files, key=self._extract_hr_number)

        target_seconds = duration_minutes * 60.0
        total_duration = 0.0
        output_parts = []
        time_offset = 0.0

        for filename in label_files:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)

            remaining = target_seconds - total_duration
            if remaining <= 0:
                break

            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            # Truncate last file to match the remaining duration.
            truncated = df.copy()
            if "Time" in truncated.columns:
                start_time = truncated["Time"].iloc[0]
                cutoff = start_time + remaining
                truncated = truncated[truncated["Time"] <= cutoff]
                if len(truncated) > 0:
                    truncated["Time"] = truncated["Time"] - start_time + time_offset
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(truncated)
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError(f"No data available for label {label}.")

        return pd.concat(output_parts, ignore_index=True)

    def concatenate_sequential_hr(self, labels: List[int], num_segments: int, output_dir: str, target_minutes: float) -> None:
        if not labels:
            raise ValueError("Labels list cannot be empty.")
        if num_segments <= 0:
            raise ValueError("num_segments must be positive.")

        os.makedirs(output_dir, exist_ok=True)

        labeled_files: List[Tuple[int, str]] = []
        for label in labels:
            self._load_label_files(label)
            for filename in self.label_files[label]:
                labeled_files.append((label, filename))

        labeled_files.sort(key=lambda item: (self._extract_hr_number(item[1]), item[0]))

        target_seconds = target_minutes * 60.0
        segments: List[Tuple[int, str]] = []

        for label, filename in labeled_files:
            segments.append((label, filename))
            if len(segments) < num_segments:
                continue

            output_df, majority_label = self._build_duration_segment(segments, target_seconds)
            if len(labels) > 1:
                final_dir = os.path.join(output_dir, f"mixed_{majority_label}")
                file_prefix = f"mixed_{majority_label}"
            else:
                final_dir = output_dir
                file_prefix = f"concat_{majority_label}"
            os.makedirs(final_dir, exist_ok=True)
            next_index = self._get_next_file_index(final_dir, file_prefix)
            file_name = f"{file_prefix}_{next_index:03d}.csv"

            output_path = os.path.join(final_dir, file_name)
            output_df.to_csv(output_path, index=False)

            segments = []

    def concatenate_random_hr(
        self,
        labels: List[int],
        files_per_segment: int,
        n_outputs: int,
        output_dir: str,
        target_minutes: float,
        allow_replacement: bool = True,
        random_seed: Optional[int] = None,
        files_per_segment_max: Optional[int] = None,
        required_majority_label: Optional[int] = None,
        min_majority_ratio: float = 0.50,
        hr_band_by_label: Optional[Dict[int, Tuple[int, int]]] = None,
        choose_unique_hr_variants: bool = True,
        adaptive_relaxation: bool = True,
    ) -> None:
        if not labels:
            raise ValueError("Labels list cannot be empty.")
        if files_per_segment <= 0:
            raise ValueError("files_per_segment must be positive.")
        if files_per_segment_max is None:
            files_per_segment_max = files_per_segment
        if files_per_segment_max < files_per_segment:
            raise ValueError("files_per_segment_max must be >= files_per_segment.")
        if n_outputs <= 0:
            raise ValueError("n_outputs must be positive.")

        rng = random.Random(random_seed)
        os.makedirs(output_dir, exist_ok=True)

        labeled_files: List[Tuple[int, str]] = []
        for label in labels:
            self._load_label_files(label)
            for filename in self.label_files[label]:
                labeled_files.append((label, filename))

        if not labeled_files:
            raise ValueError("No files available for selected labels.")

        # Group hrX/hrX_1... variants and sample one file per HR key for better diversity.
        hr_group_pool: List[Tuple[int, int, List[str]]] = []
        if choose_unique_hr_variants:
            grouped: Dict[Tuple[int, int], List[str]] = {}
            for label, filename in labeled_files:
                hr_num = self._extract_hr_number(filename)
                grouped.setdefault((label, hr_num), []).append(filename)
            hr_group_pool = [
                (label, hr_num, sorted(variants))
                for (label, hr_num), variants in grouped.items()
            ]
            if not hr_group_pool:
                raise ValueError("No HR groups available for random concatenation.")
            if files_per_segment > len(hr_group_pool):
                raise ValueError("files_per_segment is larger than number of unique HR groups.")
        if not allow_replacement and len(labeled_files) < files_per_segment:
            raise ValueError("Not enough files for sampling without replacement.")

        target_seconds = target_minutes * 60.0
        created = 0
        attempts = 0
        max_attempts = max(n_outputs * 50, 200)
        stall_attempts = 0
        max_stall_attempts = max(400, n_outputs * 10)
        current_min_majority_ratio = float(min_majority_ratio)
        current_hr_band_padding = 0.0
        reject_stats = Counter()
        existing_hashes_cache: Dict[str, set] = {}

        while created < n_outputs and attempts < max_attempts:
            attempts += 1
            if adaptive_relaxation and stall_attempts >= max_stall_attempts:
                old_ratio = current_min_majority_ratio
                current_min_majority_ratio = max(0.55, current_min_majority_ratio - 0.02)
                current_hr_band_padding = min(3.0, current_hr_band_padding + 0.5)
                stall_attempts = 0
                print(
                    f"Adaptive relaxation -> min_majority_ratio: {old_ratio:.2f} -> {current_min_majority_ratio:.2f}, "
                    f"hr_band_padding: +/-{current_hr_band_padding:.1f}"
                )
            n_files_this_output = rng.randint(files_per_segment, files_per_segment_max)
            if choose_unique_hr_variants:
                n_pick = min(n_files_this_output, len(hr_group_pool))
                selected_groups = rng.sample(hr_group_pool, n_pick)
                segments = [
                    (label, rng.choice(variants))
                    for label, _, variants in selected_groups
                ]
            else:
                if not allow_replacement and len(labeled_files) < n_files_this_output:
                    raise ValueError("Not enough files for this output when sampling without replacement.")
                if allow_replacement:
                    segments = [rng.choice(labeled_files) for _ in range(n_files_this_output)]
                else:
                    segments = rng.sample(labeled_files, n_files_this_output)

            label_counts = Counter([label for label, _ in segments])
            majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
            majority_ratio = float(label_counts[majority_label] / max(len(segments), 1))
            if required_majority_label is not None and majority_label != required_majority_label:
                reject_stats['majority_label_mismatch'] += 1
                stall_attempts += 1
                continue
            if majority_ratio < current_min_majority_ratio:
                reject_stats['majority_ratio'] += 1
                stall_attempts += 1
                continue
            if hr_band_by_label:
                band = hr_band_by_label.get(majority_label)
                if band is not None:
                    hr_values_majority = [
                        self._extract_hr_number(filename)
                        for label, filename in segments
                        if label == majority_label
                    ]
                    if not hr_values_majority:
                        reject_stats['empty_majority_hr_values'] += 1
                        stall_attempts += 1
                        continue
                    mean_hr_majority = sum(hr_values_majority) / len(hr_values_majority)
                    low = band[0] - current_hr_band_padding
                    high = band[1] + current_hr_band_padding
                    if mean_hr_majority < low or mean_hr_majority > high:
                        reject_stats['hr_band'] += 1
                        stall_attempts += 1
                        continue

            output_df, _, _ = self._build_duration_segment(segments, target_seconds)
            if len(labels) > 1:
                final_dir = os.path.join(output_dir, f"mixed_{majority_label}")
                file_prefix = f"mixed_{majority_label}"
            else:
                final_dir = output_dir
                file_prefix = f"concat_{majority_label}"
            os.makedirs(final_dir, exist_ok=True)

            if final_dir not in existing_hashes_cache:
                existing_hashes_cache[final_dir] = self._collect_hashes_in_folder(final_dir)

            content_hash = self._hash_dataframe(output_df)
            if content_hash in existing_hashes_cache[final_dir]:
                reject_stats['duplicate_hash'] += 1
                stall_attempts += 1
                continue

            next_index = self._get_next_file_index(final_dir, file_prefix)
            file_name = f"{file_prefix}_{next_index:03d}.csv"
            output_path = os.path.join(final_dir, file_name)
            output_df.to_csv(output_path, index=False)

            existing_hashes_cache[final_dir].add(content_hash)
            created += 1
            stall_attempts = 0

        print(f"Random generation done: created={created}, attempts={attempts}, skipped={attempts - created}")
        if reject_stats:
            print(f"Reject stats: {dict(reject_stats)}")
        if created < n_outputs:
            print("Warning: unable to create all requested unique outputs with current constraints.")

    def _build_duration_segment(self, segments: List[Tuple[int, str]], target_seconds: float) -> Tuple[pd.DataFrame, int, float]:
        label_counts = Counter([label for label, _ in segments])
        majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
        majority_ratio = float(label_counts[majority_label] / max(len(segments), 1))

        output_parts = []
        total_duration = 0.0
        time_offset = 0.0

        for label, filename in segments:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)
            remaining = target_seconds - total_duration
            if remaining <= 0:
                break
            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            truncated = df.copy()
            if "Time" in truncated.columns:
                start_time = truncated["Time"].iloc[0]
                cutoff = start_time + remaining
                truncated = truncated[truncated["Time"] <= cutoff]
                if len(truncated) > 0:
                    truncated["Time"] = truncated["Time"] - truncated["Time"].iloc[0]
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(self._offset_time(truncated, time_offset))
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError("Unable to build concatenated segment from provided files.")

        return pd.concat(output_parts, ignore_index=True), majority_label, majority_ratio

## Usage examples

In [51]:
from typing import Any
import math

def _hhmm_to_seconds(hhmm: str) -> int:
    parts = hhmm.strip().split(':')
    if len(parts) != 2:
        raise ValueError(f"Time '{hhmm}' must be in HH:MM format.")
    h, m = int(parts[0]), int(parts[1])
    # Accept 24:00 as end-of-day boundary for full-day plans.
    if h == 24 and m == 0:
        return 24 * 3600
    if not (0 <= h <= 23 and 0 <= m <= 59):
        raise ValueError(f"Invalid time '{hhmm}'.")
    return h * 3600 + m * 60

def _seconds_to_hhmm(seconds: int) -> str:
    if seconds < 0 or seconds > 24 * 3600:
        raise ValueError('seconds must be in [0, 86400].')
    if seconds == 24 * 3600:
        return '24:00'
    h = seconds // 3600
    m = (seconds % 3600) // 60
    return f"{h:02d}:{m:02d}"

def build_day_plan(
    start: str = '00:00',
    end: str = '24:00',
    pattern: Optional[List[Tuple[str, int]]] = None,
    segment_minutes: int = 60,
) -> List[Tuple[str, str, str, int]]:
    # ...existing code unchanged...
    start_sec = _hhmm_to_seconds(start)
    end_sec = _hhmm_to_seconds(end)
    if end_sec <= start_sec:
        raise ValueError('end must be after start.')
    if segment_minutes <= 0:
        raise ValueError('segment_minutes must be > 0.')
    step = segment_minutes * 60
    if (end_sec - start_sec) % step != 0:
        raise ValueError('Time range must be divisible by segment_minutes.')
    if not pattern:
        pattern = [
            ('pure', 0),
            ('pure', 0),
            ('mixed', 1),
            ('pure', 1),
            ('mixed', 2),
            ('pure', 2),
            ('mixed', 3),
            ('pure', 3),
            ('mixed', 2),
            ('pure', 2),
            ('mixed', 1),
            ('pure', 1),
            ('mixed', 0),
            ('pure', 0),
        ]
    for kind, level in pattern:
        if str(kind).lower() not in {'pure', 'mixed'}:
            raise ValueError(f"Invalid kind in pattern: {kind}")
        _ = int(level)
    n_segments = (end_sec - start_sec) // step
    day_plan: List[Tuple[str, str, str, int]] = []
    for i in range(n_segments):
        seg_start = start_sec + i * step
        seg_end = seg_start + step
        kind, level = pattern[i % len(pattern)]
        day_plan.append((
            _seconds_to_hhmm(seg_start),
            _seconds_to_hhmm(seg_end),
            str(kind).lower(),
            int(level),
        ))
    return day_plan

def build_explicit_day_plan(
    segments: List[Any],
    require_continuous_plan: bool = True,
) -> List[Tuple[str, str, str, int]]:
    # ...existing code unchanged...
    if not segments:
        raise ValueError('segments cannot be empty.')
    normalized = []
    for item in segments:
        if isinstance(item, dict):
            start = str(item['start'])
            end = str(item['end'])
            kind = str(item['kind']).lower()
            level = int(item['level'])
        elif isinstance(item, (list, tuple)) and len(item) == 4:
            start, end, kind, level = item
            start = str(start)
            end = str(end)
            kind = str(kind).lower()
            level = int(level)
        else:
            raise ValueError(f'Invalid segment item: {item}')
        if kind not in {'pure', 'mixed'}:
            raise ValueError(f"kind must be 'pure' or 'mixed', got: {kind}")
        start_sec = _hhmm_to_seconds(start)
        end_sec = _hhmm_to_seconds(end)
        if end_sec <= start_sec:
            raise ValueError(f'End time must be after start time: {item}')
        normalized.append({
            'start': start,
            'end': end,
            'start_sec': start_sec,
            'end_sec': end_sec,
            'kind': kind,
            'level': level,
        })
    normalized.sort(key=lambda x: x['start_sec'])
    for i in range(1, len(normalized)):
        prev = normalized[i - 1]
        cur = normalized[i]
        if cur['start_sec'] < prev['end_sec']:
            raise ValueError(
                f"Timeline overlaps between {prev['start']}-{prev['end']} and {cur['start']}-{cur['end']}."
            )
        if require_continuous_plan and cur['start_sec'] != prev['end_sec']:
            raise ValueError(
                f"Timeline is not continuous between {prev['end']} and {cur['start']}."
            )
    return [
        (item['start'], item['end'], item['kind'], item['level'])
        for item in normalized
    ]

def assign_label_column(df: pd.DataFrame, day_plan: list):
    """
    Gán cột label cho từng đoạn dựa trên day_plan.
    """
    segment_bounds = []
    offset = 0.0
    for seg in day_plan:
        if isinstance(seg, dict):
            duration = _hhmm_to_seconds(seg['end']) - _hhmm_to_seconds(seg['start'])
            label = int(seg['level'])
        else:
            _, _, _, label = seg
            duration = _hhmm_to_seconds(seg[1]) - _hhmm_to_seconds(seg[0])
        segment_bounds.append((offset, offset + duration, label))
        offset += duration
    labels = []
    for t in df['Time']:
        found = False
        for start, end, label in segment_bounds:
            if start <= t < end or (abs(t - end) < 1e-6 and t == df['Time'].iloc[-1]):
                labels.append(label)
                found = True
                break
        if not found:
            labels.append(None)
    df = df.copy()
    df['label'] = labels
    return df

# Option 1: explicit per-segment times (you can set 07:15 -> 08:15, etc.)
USE_EXPLICIT_SEGMENTS = True

EXPLICIT_SEGMENTS = [
    ('00:00', '05:00', 'pure', 0),
    ('05:00', '06:00', 'mixed', 0),
    ('06:00', '07:15', 'pure', 0),
    ('07:15', '08:15', 'mixed', 0),
    ('08:15', '09:00', 'mixed', 1),
    ('09:00', '10:00', 'pure', 1),
    ('10:00', '11:00', 'pure', 2),
    ('11:00', '12:00', 'mixed', 2),
    ('12:00', '13:30', 'pure', 1),
    ('13:30', '14:00', 'mixed', 2),
    ('14:00', '15:00', 'mixed', 3),
    ('15:00', '16:00', 'mixed', 3),
    ('16:00', '16:30', 'pure', 3),
    ('16:30', '17:30', 'pure', 3),
    ('17:30', '18:30', 'mixed', 2),
    ('18:30', '19:30', 'pure', 1),
    ('19:30', '20:00', 'mixed', 1),
    ('20:00', '21:00', 'pure', 1),
    ('21:00', '22:00', 'mixed', 0),
    ('22:00', '23:00', 'pure', 1),
    ('23:00', '24:00', 'pure', 0),
]

# Option 2: auto-build repeated pattern for a full day
START_TIME = '00:00'
END_TIME = '24:00'
SEGMENT_MINUTES = 60
PATTERN = [
    ('pure', 0),
    ('pure', 0),
    ('mixed', 1),
    ('pure', 1),
    ('mixed', 2),
    ('pure', 2),
    ('mixed', 3),
    ('pure', 3),
    ('mixed', 2),
    ('pure', 2),
    ('mixed', 1),
    ('pure', 1),
    ('mixed', 0),
    ('pure', 0),
]

if USE_EXPLICIT_SEGMENTS:
    DAY_PLAN = build_explicit_day_plan(
        segments=EXPLICIT_SEGMENTS,
        require_continuous_plan=True,
    )
    output_prefix = 'day_custom_segments'
else:
    DAY_PLAN = build_day_plan(
        start=START_TIME,
        end=END_TIME,
        pattern=PATTERN,
        segment_minutes=SEGMENT_MINUTES,
    )
    output_prefix = 'day_full_0000_2400'

daily_df, saved_path = concatenate_by_daily_scenario(
    day_plan=DAY_PLAN,
    output_dir='data/concatenated/day_scenarios',
    output_prefix=output_prefix,
    raw_base_dir='data/raw_gen',
    random_seed=None,
    require_continuous_plan=True,
    mixed_majority_ratio=0.65,
)
# Thêm cột label cho từng dòng và lưu ra file
if daily_df is not None and len(daily_df) > 0:
    daily_df = assign_label_column(daily_df, DAY_PLAN)
    daily_df.to_csv(saved_path, index=False)

display(daily_df.head())
print(f'Output file: {saved_path}')


Saved: data/concatenated/day_scenarios\day_custom_segments_010.csv | rows=22118756


,Time,Voltage,Peak,label
0,0.000000,0.954305,3,0
1,0.003906,0.967895,0,0
2,0.007812,0.887211,0,0
3,0.011719,0.544942,0,0
4,0.015625,0.466153,0,0


Output file: data/concatenated/day_scenarios\day_custom_segments_010.csv
